Solving the Heat Equation with Irksome
======================================

Let's start with the simple heat equation on $\Omega = [0,10]
\times [0,10]$, with boundary $\Gamma$:

$$
u_t - \Delta u = f
$$

$$
   u  = 0 \quad \textrm{on}\ \Gamma
$$

for some known function $f$.  At each time $t$, the solution
to this equation will be some function $u\in V$, for a suitable function
space $V$.

We transform this into weak form by multiplying by an arbitrary test function
$v\in V$ and integrating over $\Omega$.  We know have the
variational problem of finding $u:[0,T]\rightarrow V$ such
that

$$
(u_t, v) + (\nabla u, \nabla v) = (f, v)
$$


This demo implements an example used by Solin with a particular choice
of $f$ given below

As usual, we need to import Firedrake

In [8]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

try:
    import irksome
except ImportError:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git#egg=Irksome
    import irksome

from firedrake import *  # noqa: F403

A standard first time stepping method for this problem is backward Euler.  Starting from initial $u^0$, this method finds $u^{n+1} \in V$ such that
$$
\left( \frac{u^{n+1}-u^n}{\Delta t}, v \right) + \left( \nabla u^{n+1}, \nabla v \right) = \left( f^{n+1}, v \right)
$$
for all $v \in V$. 

A key contribution of Irksome is that implements the time discretization automatically by UFL manipulation.  So, users implement only the semidiscrete form of the equations and combine that with a clean description of the time discretization.

To proceed, weneed to import certain items from Irksome.  `Dt` is UFL extension that represents time differentiation, backward Euler is what it says, and `TimeStepper` is a factory function that returns an appropriately configured time stepping class.

In [2]:
from irksome import BackwardEuler, Dt, TimeStepper

In [3]:
butcher_tableau = BackwardEuler()

Now we define the mesh and piecewise linear approximating space in
standard Firedrake fashion

In [4]:
N = 100
x0 = 0.0
x1 = 10.0
y0 = 0.0
y1 = 10.0

In [5]:
msh = RectangleMesh(N, N, x1, y1)
V = FunctionSpace(msh, "CG", 1)

NameError: name 'RectangleMesh' is not defined

We define variables to store the time step size and current time value

In [ ]:
dt = Constant(10.0 / N)
t = Constant(0.0)

This defines the right-hand side using the method of manufactured solutions

In [ ]:
x, y = SpatialCoordinate(msh)
S = Constant(2.0)
C = Constant(1000.0)
B = (x-Constant(x0))*(x-Constant(x1))*(y-Constant(y0))*(y-Constant(y1))/C
R = (x * x + y * y) ** 0.5
uexact = B * atan(t)*(pi / 2.0 - atan(S * (R - t)))
rhs = Dt(uexact) - div(grad(uexact))

We define the initial condition for the fully discrete problem, which
will get overwritten at each time step

In [ ]:
u = Function(V)
u.interpolate(uexact)

Now, we will define the semidiscrete variational problem using
standard UFL notation, augmented by the ``Dt`` operator from Irksome

In [ ]:
v = TestFunction(V)
F = inner(Dt(u), v)*dx + inner(grad(u), grad(v))*dx - inner(rhs, v)*dx
bc = DirichletBC(V, 0, "on_boundary")

Later demos will show how to use Firedrake's sophisticated interface
to PETSc for efficient block solvers, but for now, we will solve the
system with a direct method.

In [ ]:
luparams = {"mat_type": "aij",
            "ksp_type": "preonly",
            "pc_type": "lu"}

Most of Irksome's magic happens in the `TimeStepper` method.  It forms a particular Runge--Kutta time stepper that transforms our semidiscrete form `F` into a fully discrete form for
the stage unknowns and sets up a variational problem to solve for the stages at each time step.

In [ ]:
stepper = TimeStepper(F, butcher_tableau, t, dt, u, bcs=bc,
                      solver_parameters=luparams)

This logic is pretty self-explanatory.  We use the
`TimeStepper`'s `TimeStepper.advance` method, which solves the variational
problem to compute the Runge-Kutta stage values and then updates the solution

In [ ]:
while (float(t) < 1.0):
    if (float(t) + float(dt) > 1.0):
        dt.assign(1.0 - t)
    stepper.advance()
    print(float(t))
    t.assign(t + dt)

Finally, we print out the relative $L^2$ error

In [ ]:
print()
print(errornorm(uexact, u)/norm(uexact))

Of course, backward Euler isn't hard to write by hand, but it's also only first order accurate.  There are many higher-order time stepping schemes, but you'd have to rewrite your UFL description for each such method.  Hence, there's no single point of entry for your problem.  Instead, Irksome gives you a wide range of possible methods.  For example, the RadauIIA family of Runge--Kutta methods generalize backward Euler to higher order.  These methods:
- are L-stable and algebraically stable
- have formal accuracy of order $2s-1$ with $s$ Runge--Kutta stages
- have stage order of $s$ (important for stiff problems like the heat equation)

We can re-do the same problem with a 2-stage (formally third order) RadauIIA method, and observe much greater accuracy.  

In [ ]:
from irksome import RadauIIA
t.assign(0)
dt.assign(10/N)
u.interpolate(uexact)
butcher_tableau = RadauIIA(2)
stepper = TimeStepper(F, butcher_tableau, t, dt, u, bcs=bc,
                      solver_parameters=luparams)
while (float(t) < 1.0):
    if (float(t) + float(dt) > 1.0):
        dt.assign(1.0 - t)
    stepper.advance()
    print(float(t))
    t.assign(t + dt)
print()
print(errornorm(uexact, u)/norm(uexact))

We were taking massive time steps, so it's not surprising that backward Euler gave a much larger error.

The mathematical properties of RadauIIA are spectacular, but there's a fly in the ointment: The algebraic system to solve at each time step couples together all of the RK stages.  It's far more complicated to write down than for backward Euler.  It's also much larger, and black-box algorithms like AMG don't seem to work well.

In the next demo, we'll look at preconditioning the large, stage-coupled system with a block triangular system.  Done correctly, this can give iteration counts counts that do not grow with the number of RK stages and allow us to make use of standard preconditioners like AMG on the blocks.